In [1]:
import os 
from dotenv import load_dotenv

load_dotenv()

if os.getenv("OPENAI_API_KEY") is None:
    raise ValueError("OPENAI_API_KEY is not set")
else:
    os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
    print("OPENAI_API_KEY is set")

OPENAI_API_KEY is set


In [2]:
from langchain_openai import ChatOpenAI

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

## PART 1 — Prompt Templates

**Task 1: PromptTemplate**
1. Create a PromptTemplate with:
   - System instruction
   - User question placeholder
2. Inject user input dynamically.
3. Test with multiple inputs.

In [4]:
from langchain_core.prompts import PromptTemplate

In [5]:
prompt = PromptTemplate.from_template(
    """
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.

    Question: {question}
    """
)

In [6]:
result = llm.invoke(prompt.format(question="What is the capital of France?"))
print(result.content)

The capital of France is Paris.


In [7]:
formatted_prompt = prompt.format(question="What is the capital of France?")
print(formatted_prompt)


    You are a helpful AI assistant.

    Answer the following question clearly and concisely.

    Question: What is the capital of France?
    


In [8]:
qsns = [
    "What is Machine Learning?",
    "What is Deep Learning?",
    "What is Natural Language Processing?",
    "What is Computer Vision?",
    "How to use LangChain?",
]

for qsn in qsns:
    print(f"Question: {qsn}")
    print(f"formatted prompt: {prompt.format(question=qsn)}")
    print("-"*100)


Question: What is Machine Learning?
formatted prompt: 
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.

    Question: What is Machine Learning?
    
----------------------------------------------------------------------------------------------------
Question: What is Deep Learning?
formatted prompt: 
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.

    Question: What is Deep Learning?
    
----------------------------------------------------------------------------------------------------
Question: What is Natural Language Processing?
formatted prompt: 
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.

    Question: What is Natural Language Processing?
    
----------------------------------------------------------------------------------------------------
Question: What is Computer Vision?
formatted prompt: 
    You are a helpful AI assistant.

    A

**Task 2: ChatPromptTemplate & Message Templates**
1. Create a ChatPromptTemplate using:
   - SystemMessage
   - HumanMessage
   - AIMessage (optional)
2. Use message prompt templates to structure conversation.
3. Compare with simple PromptTemplate.

In [18]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

system_message = SystemMessage(content="You are a helpful AI assistant.")
human_message = HumanMessage(content="What is the capital of France?")
ai_message = AIMessage(content="The capital of France is Paris.")
human_message2 = HumanMessage(content="What is the capital of India?")

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    ai_message,
    human_message2
])

In [19]:
prompt

ChatPromptTemplate(input_variables=[], input_types={}, partial_variables={}, messages=[SystemMessage(content='You are a helpful AI assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={}), AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={})])

In [20]:

messages = prompt.format_messages()
result = llm.invoke(messages)
print(result.content)


The capital of India is New Delhi.


In [21]:
messages = [
    SystemMessage(
        content="You are an expert Python tutor."
    ),
    HumanMessage(
        content="Explain decorators."
    )
]

response = llm.invoke(messages)

print(response.content)

In Python, decorators are a powerful and flexible way to modify or enhance the behavior of functions or methods. They allow you to wrap another function in order to extend its behavior without permanently modifying it. This is particularly useful for cross-cutting concerns like logging, access control, caching, and more.

### How Decorators Work

A decorator is essentially a function that takes another function as an argument, adds some functionality to it, and returns a new function. The syntax for decorators uses the `@decorator_name` syntax placed above the function definition.

### Basic Structure of a Decorator

Here's a simple example of a decorator:

```python
def my_decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper

@my_decorator
def say_hello():
    print("Hello!")

say_hello()
```

### Explanation of the Example

1. *

In [22]:
messages.append(AIMessage(content=response.content))

messages.append(
        HumanMessage(
        content="What are its advantages?"
    )
)

messages

[SystemMessage(content='You are an expert Python tutor.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Explain decorators.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='In Python, decorators are a powerful and flexible way to modify or enhance the behavior of functions or methods. They allow you to wrap another function in order to extend its behavior without permanently modifying it. This is particularly useful for cross-cutting concerns like logging, access control, caching, and more.\n\n### How Decorators Work\n\nA decorator is essentially a function that takes another function as an argument, adds some functionality to it, and returns a new function. The syntax for decorators uses the `@decorator_name` syntax placed above the function definition.\n\n### Basic Structure of a Decorator\n\nHere\'s a simple example of a decorator:\n\n```python\ndef my_decorator(func):\n    def wrapper():\n        print("Something is happening before the funct

In [23]:
res = llm.invoke(messages)
print(res.content)

Decorators in Python offer several advantages that make them a valuable tool for developers. Here are some of the key benefits:

### 1. **Code Reusability**
   - Decorators allow you to encapsulate functionality that can be reused across multiple functions or methods. This reduces code duplication and promotes the DRY (Don't Repeat Yourself) principle.

### 2. **Separation of Concerns**
   - By using decorators, you can separate the core logic of a function from its auxiliary behavior (like logging, authentication, etc.). This makes your code cleaner and easier to maintain.

### 3. **Enhanced Readability**
   - The `@decorator` syntax is clear and expressive, making it easy to see at a glance what additional behavior is being applied to a function. This improves the readability of the code.

### 4. **Flexible Functionality**
   - Decorators can be stacked, meaning you can apply multiple decorators to a single function. This allows for complex behavior to be composed in a modular way.



## PART 2 — Structured Output using Pydantic

**Task 3: Pydantic Output Schema**
1. Define a Pydantic model for chatbot response:
```
class Answer(BaseModel):
    answer: str
    confidence: float
    source: str
```
2. Use LangChain output parsers.
3. Ensure LLM output follows the schema.


In [24]:
from pydantic import BaseModel, Field

class Answer(BaseModel):
    answer: str = Field(description="The answer to the user's question")
    confidence: float = Field(description="The confidence score of the answer")
    source: str = Field(description="The source of the answer")

In [26]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=Answer)
format_instructions = parser.get_format_instructions()

In [29]:
prompt = ChatPromptTemplate(
    messages=[
        ("system", "You are a helpful assistant that can answer questions and provide information. {format_instructions}"),
        ("user", "{question}"),
    ], 
    partial_variables={"format_instructions": format_instructions}
)

prompt.format(question="What is the capital of France?", format_instructions=format_instructions)
prompt


ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"answer": {"description": "The answer to the user\'s question", "title": "Answer", "type": "string"}, "confidence": {"description": "The confidence score of the answer", "title": "Confidence", "type": "number"}, "source": {"description": "The source of the answer", "title": "Source", "type": "string"}}, "required": ["answer", "confidence", "source"]}\n```'}, messages=[SystemMessagePromptTemplate(prompt=PromptTempla

In [31]:
chain = prompt | llm | parser
res = chain.invoke({"question": "What is the LLM?"})

In [33]:
print(f"Answer: {res.answer}")
print(f"Confidence: {res.confidence}")
print(f"Source: {res.source}")

Answer: LLM stands for Large Language Model, which is a type of artificial intelligence model designed to understand and generate human language. These models are trained on vast amounts of text data and can perform various language-related tasks such as translation, summarization, and conversation.
Confidence: 0.95
Source: OpenAI


**Task 4: Validation & Error Handling**
1. Handle invalid LLM outputs.
2. Retry or fallback when schema is not followed.

In [34]:
# handle invalid LLM outputs

# retry or fallback when schema is not followed
try:
    res = chain.invoke({"question": "What is the capital of France?"})
except Exception as e:
    print(f"Error: {e}")
    res = Answer(
        answer="Unable to generate a structured answer.",
        confidence=0.0,
        source="fallback"
    )

# handle invalid LLM outputs
print(f"Answer: {res.answer}")
print(f"Confidence: {res.confidence}")
print(f"Source: {res.source}")

Answer: Paris
Confidence: 0.99
Source: General knowledge


## PART 3 — Chains in LangChain

**Task 5: Simple Chain**
1. Create a basic chain:
Prompt → Any LLM → Output
2. Execute the chain.


In [36]:
from langchain_core.output_parsers import StrOutputParser

In [35]:
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful AI tutor.

    Explain the following topic in simple terms:

    {topic}
    """
)

In [37]:
parser = StrOutputParser()
chain = prompt | llm | parser

In [38]:
res = chain.invoke({"topic": "Transformers"})
print(res)


Sure! Let's break down the concept of Transformers in a simple way.

### What are Transformers?

Transformers are a type of model used in machine learning, especially in natural language processing (NLP). They help computers understand and generate human language. Think of them as a very smart way for computers to read and write text.

### How do Transformers Work?

1. **Attention Mechanism**: The key feature of Transformers is something called "attention." Imagine you're reading a book. When you read a sentence, you might pay more attention to certain words that are important for understanding the meaning. Transformers do something similar. They look at all the words in a sentence and decide which ones are most important for understanding the context.

2. **Input and Output**: When you give a Transformer a sentence, it breaks it down into smaller parts (like words or subwords). It then processes these parts to understand their relationships and meanings. After processing, it can gener


**Task 6: Conditional Chain**
1. Create a chain that:
   - Uses retrieval if question is factual
   - Uses direct LLM response otherwise
2. Implement conditional logic.

In [50]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough

In [51]:
splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)

docs = splitter.split_text(res)

chroma_client = Chroma.from_texts(
    texts=docs,
    embedding=OpenAIEmbeddings(),
    persist_directory="chroma_db"
)

chroma_retriever = chroma_client.as_retriever()

Created a chunk of size 239, which is longer than the specified 200
Created a chunk of size 393, which is longer than the specified 200
Created a chunk of size 299, which is longer than the specified 200
Created a chunk of size 228, which is longer than the specified 200
Created a chunk of size 303, which is longer than the specified 200
Created a chunk of size 219, which is longer than the specified 200


In [55]:
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

prompt = ChatPromptTemplate.from_template("""
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.
    Use ONLY the context below. If unsure, say you don't know.
    Context: {context}

    Question: {question}
    """)

retriever_chain = {
    "context":  chroma_retriever | format_docs,
    "question": RunnablePassthrough()
} | prompt | llm | parser


In [58]:
res = retriever_chain.invoke("What is the Transformer?")
print(res)


Transformers are a type of model used in machine learning, particularly in natural language processing. They are designed to handle sequential data and are known for their ability to process information in parallel, which makes them efficient for tasks like translation and text generation.


In [59]:
res = retriever_chain.invoke("What is the capital of France?")
print(res)

I don't know.


In [60]:
def is_factual(question):

    factual_keywords = [
        "when",
        "where",
        "who",
        "what",
        "how many",
        "date",
        "year"
    ]

    question_lower = question.lower()

    return any(
        keyword in question_lower
        for keyword in factual_keywords
    )

In [61]:
simple_prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.

    Question: {question}
    """
)

chain = simple_prompt | llm | parser

res = chain.invoke({"question": "What is the capital of France?"})
print(res)

The capital of France is Paris.


In [62]:
from langchain_core.runnables import RunnableBranch

In [64]:
branch = RunnableBranch(
    (is_factual, retriever_chain),
    chain
)


In [65]:

res = branch.invoke("What is Transformer?")
print(res)

Transformers are a type of model used in machine learning and natural language processing, designed to handle sequential data and improve the efficiency of processing such data.


In [66]:
res2 = branch.invoke("write a poem on transformer")
print(res2)

**Ode to the Transformer**

In the heart of the circuit, where currents flow,  
A marvel of engineering, a silent glow.  
With coils entwined in a dance of fate,  
Transforming the power, it holds the weight.  

From high to low, it bends the might,  
Voltage whispers, turning day to night.  
A guardian of energy, steadfast and true,  
In every home, it quietly renews.  

Copper and iron, in harmony blend,  
A symphony of science, on which we depend.  
With hums and vibrations, it works with grace,  
A bridge of connection in this electric space.  

So here's to the transformer, both humble and grand,  
A silent sentinel, across the land.  
In every flicker, in every spark,  
It lights up our lives, igniting the dark.  



**Task 7: Parallel Chain**
1. Run multiple chains in parallel:
   - One chain generates answer
   - One chain generates summary
   - One chain generates follow-up questions
2. Combine outputs.

In [44]:
from langchain_core.runnables import RunnableParallel

In [67]:
answer_prompt = ChatPromptTemplate.from_template(
    """
Answer this question:

{question}
"""
)

answer_chain = (
    answer_prompt
    | llm
    | StrOutputParser()
)

In [68]:
summary_prompt = ChatPromptTemplate.from_template(
    """
Give a short summary of the following question:

{question}
"""
)

summary_chain = (
    summary_prompt
    | llm
    | StrOutputParser()
)


In [69]:
followup_prompt = ChatPromptTemplate.from_template(
    """
Generate 3 useful follow-up questions
based on this question:

{question}
"""
)

followup_chain = (
    followup_prompt
    | llm
    | StrOutputParser()
)

In [70]:
parallel_chain = RunnableParallel(
    answer=answer_chain,
    summary=summary_chain,
    followups=followup_chain
)

In [71]:
result = parallel_chain.invoke({
    "question": "Explain Retrieval Augmented Generation."
})

print(result['answer'])
print(result['summary'])
print(result['followups'])

Retrieval-Augmented Generation (RAG) is a hybrid approach that combines the strengths of information retrieval and generative models to improve the quality and relevance of generated text. This method is particularly useful in tasks where the generation of text needs to be grounded in factual information or specific knowledge that may not be contained within the model's training data.

### Key Components of RAG:

1. **Retrieval Component**: 
   - This part of the system is responsible for fetching relevant documents or pieces of information from a large corpus or database. It typically uses techniques from information retrieval, such as vector similarity search or traditional keyword-based search, to identify the most pertinent data based on the input query.

2. **Generative Component**: 
   - After retrieving relevant documents, the generative model (often based on architectures like Transformers) takes this information and uses it to produce coherent and contextually appropriate text

## PART 4 — Runnables & LCEL

**Task 8: Runnables Basics**
1. Convert chains into Runnables.
2. Use RunnablePassthrough.
3. Compose components using LCEL syntax (|).

In [72]:
prompt = ChatPromptTemplate.from_template(
    "Explain {topic}"
)

chain = prompt | llm | StrOutputParser()

In [73]:
res = chain.invoke({"topic": "Transformers"})
print(res)

Transformers are a type of neural network architecture that has revolutionized the field of natural language processing (NLP) and has also been applied to various other domains, including computer vision and audio processing. Introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017, Transformers have become the foundation for many state-of-the-art models, such as BERT, GPT, and T5.

### Key Components of Transformers

1. **Self-Attention Mechanism**:
   - The core innovation of Transformers is the self-attention mechanism, which allows the model to weigh the importance of different words in a sentence relative to each other. This means that each word can attend to every other word in the input sequence, enabling the model to capture contextual relationships effectively.

2. **Positional Encoding**:
   - Since Transformers do not have a built-in notion of sequence order (unlike recurrent neural networks), they use positional encodings to inject information about the

In [74]:
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

prompt = ChatPromptTemplate.from_template("""
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.
    Use ONLY the context below. If unsure, say you don't know.
    Context: {context}

    Question: {question}
    """)

retriever_chain = {
    "context":  chroma_retriever | format_docs,
    "question": RunnablePassthrough()
} | prompt | llm | parser


In [76]:
res = retriever_chain.invoke("what is attention?")
print(res)

Attention is a mechanism used in Transformers that allows the model to focus on certain words in a sentence that are important for understanding the context, similar to how a reader pays more attention to key words while reading.


**Task 9: LCEL-Based RAG Chain**
Build an LCEL pipeline:
Retriever | Prompt | LLM | Output Parser
Test with multiple user queries.


In [77]:
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

prompt = ChatPromptTemplate.from_template("""
    You are a helpful AI assistant.

    Answer the following question clearly and concisely.
    Use ONLY the context below. If unsure, say you don't know.
    Context: {context}

    Question: {question}
    """)

retriever_chain = {
    "context":  chroma_retriever | format_docs,
    "question": RunnablePassthrough()
} | prompt | llm | parser


In [78]:
qsns = [
    "What is Transformer?",
    "What is Attention?",
    "What are the layers in Transformer?",
    "Why Transformer is important?",
    "What is the future of Transformer?"
]

for qsn in qsns:
    res = retriever_chain.invoke(qsn)
    print(res)
    

Transformers are a type of model used in machine learning, particularly in natural language processing, that allows for the processing of data in parallel and captures relationships in sequences effectively.
Attention is a mechanism used in Transformers that allows the model to focus on certain words in a sentence that are most important for understanding the context, similar to how a reader pays more attention to key words while reading.
The layers in a Transformer are multiple levels that refine the understanding of the input. Each layer adds complexity to the understanding, similar to how each layer of a cake adds more flavor and texture.
Transformers are important because they enable computers to understand and generate human language effectively, leading to significant advancements in technology and language processing.
I don't know.


**Task 10: Observations & Insights**
Write short answers:

1. Why structured output is important → free-form text is messy to use in code (APIs, UI, tools). Pydantic / parsers force a fixed shape (`answer`, `confidence`, `source`) so downstream code doesn’t break on weird LLM wording.
2. Advantages of LCEL over traditional chains → pipe with `|` (`prompt | llm | parser`), mix dicts / `RunnableParallel` / `RunnableBranch`, easier to compose and reuse than old LLMChain glue. feels like normal Python dataflow.
3. When to use parallel vs conditional chains → parallel (`RunnableParallel`) when you want several outputs at once (answer + summary + follow-ups). conditional (`RunnableBranch`) when the *path* should change based on the input (e.g. factual → retrieve, otherwise → direct LLM).
